# VOICEGUARD — Baseline Deepfake Detector Experiment

This notebook walks through loading a directory-based dataset, extracting baseline
features (log-mel + MFCC summaries), training a logistic-regression classifier with a
speaker-independent split, and evaluating it (accuracy, precision/recall, EER).

Populate `datasets/real/` and `datasets/synthetic/` with your own legally obtained audio
before running this notebook — no data is downloaded automatically.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from ml.deepfake.dataset import load_directory_dataset, speaker_independent_split, dataset_statistics
from ml.deepfake.preprocessing import build_feature_matrix

items = load_directory_dataset(Path('../datasets/real'), Path('../datasets/synthetic'))
print(dataset_statistics(items))

In [ ]:
train_items, val_items, test_items = speaker_independent_split(items)
print(f"train={len(train_items)} val={len(val_items)} test={len(test_items)}")

X_train, y_train, kept_train = build_feature_matrix(train_items)
X_test, y_test, kept_test = build_feature_matrix(test_items)
print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

if len(set(y_train.tolist())) < 2:
    print('Need both classes in the training split to fit a model -- add more data.')
else:
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train, y_train)
    if len(y_test) > 0:
        y_pred = model.predict(X_test)
        print(classification_report(y_test, y_pred, target_names=['real', 'synthetic']))
    else:
        print('No test samples available yet.')